# Linear Regression: Mathematics and Gradient Descent

**Teaching guide — from a straight line to learning its coefficients**

Linear regression looks for the straight line that best fits the data. Here, “best” means minimizing the average squared prediction error. We will first learn the coefficients through gradient descent, then derive the direct least-squares solution.

Author:
Manuel Eugenio Morocho Cayamcela, PhD

## 1. One feature and one value to predict

Use this small dataset on the board:

| Feature $x_i$ | Actual value $y_i$ |
|---|---|
| 0 | 1 |
| 1 | 3 |
| 2 | 5 |

Each row is one observation. The subscript $i$ identifies a row, and $n=3$ is the number of observations.

Our model is:

$$\boxed{\hat y_i=\beta_0+\beta_1x_i}$$

- $x_i$: the observed input feature.
- $y_i$: the observed target, which we want to predict.
- $\hat y_i$: the model's prediction. The hat distinguishes a prediction from an observation.
- $\beta_0$: the **intercept**, or prediction when $x=0$.
- $\beta_1$: the **slope**, or change in the prediction for a one-unit increase in $x$.

The data $(x_i,y_i)$ stay fixed during training. The coefficients $\beta_0$ and $\beta_1$ are what we change.

For these particular data, the perfect line is $\hat y=1+2x$. We can see the answer, but we will discover it through repeated updates, as we would when the answer is unknown.

**Ask the class:** What does the model predict at $x=2$ if $\beta_0=1$ and $\beta_1=2$? Substitute into the equation: $1+2(2)=5$.

## 2. Start with an initial line

Initialize both coefficients at zero:

$$\boxed{\beta_0=0,\qquad\beta_1=0}$$

The initial line is $\hat y=0$: it predicts zero for every input.

Zero initialization works for linear regression; random initialization is not required. The loss is a convex quadratic in the coefficients, so there are no nonglobal local minima. In this example, the feature values vary, so the least-squares solution is unique.

**Board reminder:** Initial values are a starting guess, not values calculated from a fitted model.

## 3. Turn prediction errors into one loss

Define the signed prediction error for observation $i$ as:

$$e_i=\hat y_i-y_i=\beta_0+\beta_1x_i-y_i.$$

A positive error means overprediction; a negative error means underprediction. Some texts define a residual as $y_i-\hat y_i$, which has the opposite sign. We use $e_i=\hat y_i-y_i$ consistently in this derivation.

Simply adding signed errors is misleading: an error of $+3$ and an error of $-3$ cancel, even though both predictions are wrong. Squaring avoids cancellation and gives larger errors greater weight.

The **mean squared error** is:

$$\operatorname{MSE}(\beta_0,\beta_1)=\frac{1}{n}\sum_{i=1}^{n}e_i^2.$$

For the derivation, use half the MSE as the loss:

$$\boxed{L(\beta_0,\beta_1)=\frac{1}{2n}\sum_{i=1}^{n}(\beta_0+\beta_1x_i-y_i)^2}.$$

Here, $\sum$ means “add the expression for every observation.” For our three rows:

$$L=\frac{1}{6}\left[(\beta_0-1)^2+(\beta_0+\beta_1-3)^2+(\beta_0+2\beta_1-5)^2\right].$$

The factor $1/2$ simplifies derivatives. Multiplying the objective by a positive constant does not change its minimizer. However, it does scale its gradients: using MSE instead of $L$ with the same learning rate would double each update.

## 4. Derivatives: what changes when a coefficient changes?

A derivative measures the local rate of change of a function. Here we want to know how the loss changes when we change a coefficient slightly.

Because $L$ depends on two coefficients, we use **partial derivatives**:

- $\frac{\partial L}{\partial\beta_0}$: vary $\beta_0$ while holding $\beta_1$ fixed.
- $\frac{\partial L}{\partial\beta_1}$: vary $\beta_1$ while holding $\beta_0$ fixed.

The symbol $\partial$ indicates a partial derivative. It is not an additional variable.

Useful differentiation rules:

| Rule | Example |
|---|---|
| A constant has derivative zero | $\frac{d}{dt}7=0$ |
| A variable has derivative one | $\frac{d}{dt}t=1$ |
| A constant multiplier stays outside | $\frac{d}{dt}(ct)=c$ |
| Power rule | $\frac{d}{dt}t^2=2t$ |
| Differentiate a sum term by term | $\frac{d}{dt}[f(t)+g(t)]=f'(t)+g'(t)$ |

During training, $x_i$ and $y_i$ are constants with respect to the coefficients. Therefore:

$$\frac{\partial e_i}{\partial\beta_0}=1+0-0=1,$$

$$\frac{\partial e_i}{\partial\beta_1}=0+x_i-0=x_i.$$

**Ask the class:** Why is the second derivative $x_i$, not $\beta_1$? Because we differentiate with respect to $\beta_1$, while $x_i$ is the constant multiplying it.

## 5. The chain rule: differentiate the outside, then the inside

We cannot differentiate $e_i^2$ as though $e_i$ were independent of the coefficient. The error itself changes when the coefficient changes.

The **chain rule** handles a function inside another function. If $u=g(t)$ and $f$ acts on $u$, then:

$$\boxed{\frac{d}{dt}f(g(t))=f'(g(t))\,g'(t)}.$$

In words: **differentiate the outer function, keep the inner expression inside it, then multiply by the derivative of the inner expression.**

For a squared error:

- Inner function: $u=e_i=\beta_0+\beta_1x_i-y_i$.
- Outer function: $f(u)=u^2$.
- Outer derivative: $f'(u)=2u=2e_i$.

Thus, for either coefficient $\beta_j$:

$$\frac{\partial(e_i^2)}{\partial\beta_j}=2e_i\frac{\partial e_i}{\partial\beta_j}.$$

The dependency is **coefficient → prediction → error → squared error → total loss**.

For a concrete row, take $x_i=2$ and $y_i=5$:

$$e_i^2=(\beta_0+2\beta_1-5)^2.$$

With respect to the slope:

$$\frac{\partial}{\partial\beta_1}(\beta_0+2\beta_1-5)^2
=2(\beta_0+2\beta_1-5)\cdot2.$$

The first $2$ comes from differentiating the square. The last $2$ comes from differentiating the inside expression with respect to $\beta_1$, because this row has $x_i=2$.

With respect to the intercept:

$$\frac{\partial}{\partial\beta_0}(\beta_0+2\beta_1-5)^2
=2(\beta_0+2\beta_1-5)\cdot1.$$

**Common mistake:** Writing only $2e_i$ for the slope derivative omits the inner derivative $x_i$.

## 6. Derive the two loss gradients

The factor $1/(2n)$ is constant, so it stays outside the derivative. Differentiate each term of the sum and apply the chain rule.

### Intercept

$$\begin{aligned}
\frac{\partial L}{\partial\beta_0}
&=\frac{1}{2n}\sum_{i=1}^{n}\frac{\partial(e_i^2)}{\partial\beta_0}\\
&=\frac{1}{2n}\sum_{i=1}^{n}2e_i\frac{\partial e_i}{\partial\beta_0}\\
&=\frac{1}{2n}\sum_{i=1}^{n}2e_i\cdot1\\
&=\boxed{\frac{1}{n}\sum_{i=1}^{n}e_i}.
\end{aligned}$$

### Slope

$$\begin{aligned}
\frac{\partial L}{\partial\beta_1}
&=\frac{1}{2n}\sum_{i=1}^{n}\frac{\partial(e_i^2)}{\partial\beta_1}\\
&=\frac{1}{2n}\sum_{i=1}^{n}2e_i\frac{\partial e_i}{\partial\beta_1}\\
&=\frac{1}{2n}\sum_{i=1}^{n}2e_i x_i\\
&=\boxed{\frac{1}{n}\sum_{i=1}^{n}e_i x_i}.
\end{aligned}$$

The $2$ from the square cancels the $2$ in the loss denominator. This is why we chose half the MSE.

The **gradient** collects both partial derivatives:

$$\nabla L=\begin{bmatrix}\partial L/\partial\beta_0\\\partial L/\partial\beta_1\end{bmatrix}.$$

The intercept gradient is the average error. The slope gradient is the average of errors multiplied by their input values.

## 7. Why gradient descent subtracts the gradient

A positive derivative means increasing that coefficient slightly increases the loss locally. A negative derivative means increasing it slightly decreases the loss locally. To move downhill, we take a step opposite to the gradient.

For a small change $\Delta\boldsymbol\beta$, the first-order approximation is:

$$L(\boldsymbol\beta+\Delta\boldsymbol\beta)\approx L(\boldsymbol\beta)+\nabla L^T\Delta\boldsymbol\beta.$$

Choosing $\Delta\boldsymbol\beta=-\alpha\nabla L$ gives:

$$L(\boldsymbol\beta-\alpha\nabla L)\approx L(\boldsymbol\beta)-\alpha\left[\left(\frac{\partial L}{\partial\beta_0}\right)^2+\left(\frac{\partial L}{\partial\beta_1}\right)^2\right].$$

For a sufficiently small positive step size, this points toward a lower loss. A step that is too large can overshoot and increase the loss.

The update rules are:

$$\boxed{\beta_0^{(t+1)}=\beta_0^{(t)}-\alpha\frac{1}{n}\sum_{i=1}^{n}e_i^{(t)}}$$

$$\boxed{\beta_1^{(t+1)}=\beta_1^{(t)}-\alpha\frac{1}{n}\sum_{i=1}^{n}e_i^{(t)}x_i}.$$

Here, $(t)$ is the iteration index, not an exponent. The learning rate $\alpha$ controls the step size. Use **$\alpha=0.3$** for this dataset and this loss definition; it is not a universal choice.

**Both coefficients must use the same errors, computed from the old coefficients.** Calculate both gradients before changing either coefficient. Since every update uses all three observations, this is **batch gradient descent**.

## 8. First update, calculated by hand

Start with $\beta_0=0$ and $\beta_1=0$.

| $x_i$ | $y_i$ | $\hat y_i$ | $e_i=\hat y_i-y_i$ | $e_ix_i$ |
|---|---|---|---|---|
| 0 | 1 | 0 | −1 | 0 |
| 1 | 3 | 0 | −3 | −3 |
| 2 | 5 | 0 | −5 | −10 |
| **Sum** | | | **−9** | **−13** |

The initial loss is:

$$L=\frac{(-1)^2+(-3)^2+(-5)^2}{6}=\frac{35}{6}\approx5.8333.$$

Compute both gradients:

$$\frac{\partial L}{\partial\beta_0}=\frac{-9}{3}=-3,\qquad
\frac{\partial L}{\partial\beta_1}=\frac{-13}{3}.$$

Then update:

$$\beta_0^{(1)}=0-0.3(-3)=\boxed{0.9},$$

$$\beta_1^{(1)}=0-0.3\left(\frac{-13}{3}\right)=\boxed{1.3}.$$

The new line is:

$$\boxed{\hat y=0.9+1.3x}.$$

**Ask the class:** Why do both coefficients increase? Both gradients are negative, and subtracting a negative number increases the coefficient.

## 9. Second update, using new predictions

Now use $\beta_0=0.9$ and $\beta_1=1.3$ to recompute every prediction and error.

| $x_i$ | $y_i$ | $\hat y_i$ | $e_i$ | $e_ix_i$ |
|---|---|---|---|---|
| 0 | 1 | 0.9 | −0.1 | 0 |
| 1 | 3 | 2.2 | −0.8 | −0.8 |
| 2 | 5 | 3.5 | −1.5 | −3.0 |
| **Sum** | | | **−2.4** | **−3.8** |

Before this update, the loss is:

$$L=\frac{0.01+0.64+2.25}{6}=\frac{2.90}{6}\approx0.4833.$$

The next coefficients are:

$$\beta_0^{(2)}=0.9-0.3\left(\frac{-2.4}{3}\right)=\boxed{1.14},$$

$$\beta_1^{(2)}=1.3-0.3\left(\frac{-3.8}{3}\right)=\boxed{1.68}.$$

The new line is:

$$\boxed{\hat y=1.14+1.68x}.$$

Its predictions are $1.14$, $2.82$, and $4.50$, so its errors are $0.14$, $-0.18$, and $-0.50$:

$$L=\frac{0.14^2+(-0.18)^2+(-0.50)^2}{6}=\frac{0.302}{6}\approx0.0503.$$

| State | Intercept $\beta_0$ | Slope $\beta_1$ | Loss $L$ |
|---|---|---|---|
| Initial | 0 | 0 | 5.8333 |
| After the first update | 0.90 | 1.30 | 0.4833 |
| After the second update | 1.14 | 1.68 | 0.0503 |
| Ideal solution | 1 | 2 | 0 |

Each loss in the table is evaluated at the coefficients on that same row. MSE would be twice the listed loss.

## 10. Repeat and decide when to stop

Repeat **predict → calculate errors → calculate gradients → update both coefficients**.

Stop when the loss changes very little, the gradient is sufficiently small, or a maximum number of iterations is reached. A small loss change alone can also result from an excessively small learning rate, so inspect the overall progress.

For this example, $\alpha=0.3$ makes the coefficients converge to $\beta_0=1$ and $\beta_1=2$.

> The intercept moves the line up or down according to the average error. The slope changes its tilt according to errors weighted by the input values. Together, they reduce the squared error.

**A coefficient does not have to move closer to its final value at every step.** The intercept moves from $0.9$ to $1.14$, passing its ideal value of $1$, while the total loss decreases. The coefficients must be considered together.

The following code reproduces the hand calculations and continues the updates. It uses the same loss and simultaneous-update rule.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([0.0, 1.0, 2.0])
Y = np.array([1.0, 3.0, 5.0])
beta_0 = 0.0
beta_1 = 0.0
alpha = 0.3
max_iterations = 1000
tolerance = 1e-12

# Store coefficients and loss at the same point in each iteration.
history = [(0, beta_0, beta_1, np.mean((beta_0 + beta_1 * X - Y) ** 2) / 2)]
for iteration in range(1, max_iterations + 1):
    predictions = beta_0 + beta_1 * X
    e = predictions - Y

    # Compute BOTH gradients with the OLD coefficients.
    gradient_beta_0 = np.mean(e)
    gradient_beta_1 = np.mean(e * X)
    beta_0, beta_1 = (
        beta_0 - alpha * gradient_beta_0,
        beta_1 - alpha * gradient_beta_1,
    )

    loss = np.mean((beta_0 + beta_1 * X - Y) ** 2) / 2
    history.append((iteration, beta_0, beta_1, loss))
    if abs(history[-2][3] - loss) < tolerance:
        break

print('Iteration | beta_0 | beta_1 | L (half-MSE)')
for step, intercept, slope, loss in history[:3]:
    print(f'{step:9d} | {intercept:6.2f} | {slope:6.2f} | {loss:.4f}')
print(f'Final: beta_0={beta_0:.6f}, beta_1={beta_1:.6f}, L={history[-1][3]:.3e}')
print(f'Updates completed: {iteration}')

## 11. See the line move

Compare the initial line, the first two updates, and the final learned line. In the second plot, observe how the loss decreases over the updates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
X_line = np.linspace(-0.25, 2.25, 100)
axes[0].scatter(X, Y, color='black', label='Observed data', zorder=5)
for step, intercept, slope, loss in [history[0], history[1], history[2], history[-1]]:
    axes[0].plot(X_line, intercept + slope * X_line,
                 label=f'Update {step}: beta_0={intercept:.2f}, beta_1={slope:.2f}')
axes[0].set(xlabel='Feature x', ylabel='Target y', title='Learning the regression line')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)
axes[1].semilogy([row[0] for row in history], [row[3] for row in history])
axes[1].set(xlabel='Number of updates', ylabel='L (half-MSE, logarithmic scale)',
            title='Loss during gradient descent')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Derive the direct least-squares solution

Gradient descent approaches the minimum through updates. For simple linear regression, we can also find the minimum by setting both derivatives to zero and solving the resulting equations.

From the intercept derivative:

$$0=\sum_{i=1}^{n}(\beta_0+\beta_1x_i-y_i)
=n\beta_0+\beta_1\sum_{i=1}^{n}x_i-\sum_{i=1}^{n}y_i.$$

Divide by $n$ and define the sample means $\bar x=\frac{1}{n}\sum x_i$ and $\bar y=\frac{1}{n}\sum y_i$:

$$\boxed{\beta_0=\bar y-\beta_1\bar x}.$$

From the slope derivative:

$$0=\sum_{i=1}^{n}(\beta_0+\beta_1x_i-y_i)x_i.$$

Substitute the intercept expression:

$$0=\sum_{i=1}^{n}\left[\beta_1(x_i-\bar x)-(y_i-\bar y)\right]x_i.$$

Rearrange:

$$\beta_1\sum_{i=1}^{n}(x_i-\bar x)x_i
=\sum_{i=1}^{n}(y_i-\bar y)x_i.$$

Why can we center the remaining $x_i$ terms? Write $x_i=(x_i-\bar x)+\bar x$. Since $\sum(x_i-\bar x)=0$ and $\sum(y_i-\bar y)=0$, the extra terms involving $\bar x$ vanish. Therefore:

$$\boxed{\beta_1=\frac{\sum_{i=1}^{n}(x_i-\bar x)(y_i-\bar y)}{\sum_{i=1}^{n}(x_i-\bar x)^2}},\qquad
\boxed{\beta_0=\bar y-\beta_1\bar x}.$$

These zero-gradient equations are called the **normal equations**. Because this loss is convex, their solution is a global minimum. The denominator must be nonzero: if all $x_i$ are identical, the slope and intercept cannot be uniquely identified separately.

### Apply the formulas to the same three observations

$$\bar x=\frac{0+1+2}{3}=1,\qquad\bar y=\frac{1+3+5}{3}=3.$$

| $x_i$ | $y_i$ | $x_i-\bar x$ | $y_i-\bar y$ | $(x_i-\bar x)(y_i-\bar y)$ | $(x_i-\bar x)^2$ |
|---|---|---|---|---|---|
| 0 | 1 | −1 | −2 | 2 | 1 |
| 1 | 3 | 0 | 0 | 0 | 0 |
| 2 | 5 | 1 | 2 | 2 | 1 |
| **Sum** | | | | **4** | **2** |

$$\beta_1=\frac{4}{2}=2,\qquad\beta_0=3-2(1)=1.$$

The direct solution and converged gradient descent produce the same line, $\hat y=1+2x$. The intercept predicts $1$ at $x=0$; the slope adds $2$ to the prediction for every additional unit of $x$.

In [ ]:
# Direct least-squares solution for the same observations
X_mean = np.mean(X)
Y_mean = np.mean(Y)
beta_1_direct = np.sum((X - X_mean) * (Y - Y_mean)) / np.sum((X - X_mean) ** 2)
beta_0_direct = Y_mean - beta_1_direct * X_mean
print(f'Direct solution: beta_0={beta_0_direct:.2f}, beta_1={beta_1_direct:.2f}')
print(f'Gradient descent: beta_0={beta_0:.6f}, beta_1={beta_1:.6f}')

## 13. Extend the same reasoning to multiple features

With $p$ features, observation $i$ has inputs $x_{i1},\ldots,x_{ip}$:

$$\hat y_i=\beta_0+\sum_{j=1}^{p}\beta_jx_{ij},\qquad e_i=\hat y_i-y_i.$$

The loss remains $L=\frac{1}{2n}\sum e_i^2$. The same chain rule gives:

$$\frac{\partial L}{\partial\beta_0}=\frac{1}{n}\sum_{i=1}^{n}e_i,\qquad
\frac{\partial L}{\partial\beta_j}=\frac{1}{n}\sum_{i=1}^{n}e_i x_{ij}\quad(j=1,\ldots,p).$$

Each slope uses errors weighted by its own feature. All coefficients are updated from the same old predictions. The fitted surface is a plane with two features and a hyperplane with more features.

The companion medical-expenses notebook applies multiple linear regression using scikit-learn. Its `LinearRegression` estimator solves ordinary least squares using numerical linear algebra; it does not run the gradient-descent loop shown here. Gradient descent is used in this guide to make learning the coefficients explicit.

## 14. Teaching checkpoints

Before moving to the medical-expenses example, ask students to explain:

1. Which quantities stay fixed during training, and which are learned?
2. Why do we square errors instead of adding signed errors?
3. In the chain rule, what are the outer and inner functions?
4. Why does the slope gradient contain $x_i$, while the intercept gradient does not?
5. Why do we subtract the gradient, and what could happen if the learning rate is too large?
6. Why must both gradients be calculated before either coefficient changes?
7. Why can the intercept move past $1$ while the total loss still decreases?
8. If we use MSE instead of half-MSE, how should the learning rate change to preserve the same updates? **Halve it: use $0.15$ instead of $0.3$.**

For a board exercise, have students calculate the third update from $\beta_0=1.14$ and $\beta_1=1.68$. The errors are $0.14$, $-0.18$, and $-0.50$; their sum is $-0.54$, and the sum of $e_ix_i$ is $-1.18$. The next coefficients are **$\beta_0=1.194$ and $\beta_1=1.798$**.